# CEG-WM Colab 完整工作流执行指南

本 Notebook 在 Google Colab 上执行完整的机器学习工作流（embed/detect/calibrate/evaluate），生成新的 run_root 目录并下载最终结果。

**执行流水线**：
- **Embed 阶段**：对输入进行特征提取和防水印嵌入
- **Detect 阶段**：检测防水印信号
- **Calibrate 阶段**：校准检测阈值和概率
- **Evaluate 阶段**：评估整个工作流的性能

## 第 1 步：上传和解压代码包

将 CEG-WM 代码上传到 Colab。

**方式：从本地上传**
- 直接在 Colab UI 中上传 CEG-WM.zip

In [ ]:
import sys
import os
from pathlib import Path
import psutil

# 检测 Google Colab 环境
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("=" * 80)
print("初始化 Notebook 环境")
print("=" * 80)

# 设置工作目录
if IN_COLAB:
    WORK_DIR = Path("/tmp/ceg_wm_workspace")
    print(f"✓ 检测到 Google Colab 环境")
else:
    WORK_DIR = Path.cwd()
    print(f"✓ 本地 Jupyter 环境")

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"  工作目录：{WORK_DIR}")

# 显示系统信息
print("\n系统信息：")
print(f"  操作系统：{sys.platform}")
print(f"  Python 版本：{sys.version.split()[0]}")
print(f"  CPU 核心数：{psutil.cpu_count()}")
print(f"  总内存：{psutil.virtual_memory().total / (1024**3):.1f} GB")
print(f"  可用内存：{psutil.virtual_memory().available / (1024**3):.1f} GB")

print("\n✓ 环境初始化完成")

In [ ]:

import zipfile
from google.colab import files
import os

# 1. 上传 zip 文件
print("请上传您的 CEG-WM.zip 文件。")
uploaded = files.upload()

for fn in uploaded.keys():
    print(f'已上传文件："{fn}"，大小 {len(uploaded[fn]) / (1024*1024):.2f} MB')
    zip_filename = fn

    # 2. 解压 zip 文件
    if zip_filename.endswith('.zip'):
        try:
            with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
                zip_ref.extractall(str(WORK_DIR))
            print(f"✓ 文件 '{zip_filename}' 已成功解压缩到 {WORK_DIR}")
        except zipfile.BadZipFile:
            print(f"✗ 错误：文件 '{zip_filename}' 不是有效的 ZIP 文件。")
        except Exception as e:
            print(f"✗ 解压缩时发生错误: {e}")
    else:
        print(f"⚠ 文件 '{zip_filename}' 不是 ZIP 文件。跳过解压缩。")


In [ ]:

# 验证和定位 CEG-WM 代码库
from pathlib import Path

def find_ceg_wm_root(base_dir):
    """
    功能：查找 CEG-WM 根目录（智能路径发现）。
    """
    base_dir = Path(base_dir)
    
    # 检查：特征目录结构
    characteristic_dirs = ["main/cli", "configs", "scripts", "requirements.txt"]
    
    # 首先检查直接目录
    if all((base_dir / d).exists() for d in characteristic_dirs):
        return base_dir
    
    # 查找包含正确结构的子目录
    for subdir in sorted(base_dir.iterdir()):
        if subdir.is_dir() and not subdir.name.startswith('.'):
            if all((subdir / d).exists() for d in characteristic_dirs):
                return subdir
    
    # 尝试找任何包含 main/cli 的目录（最后的备选方案）
    for possible_root in base_dir.rglob("main"):
        if possible_root.is_dir():
            ceg_root = possible_root.parent
            if (ceg_root / "configs").exists() and (ceg_root / "scripts").exists():
                return ceg_root
    
    raise FileNotFoundError(
        f"无法找到 CEG-WM 根目录\n"
        f"  搜索路径：{base_dir}\n"
        f"  期望目录结构：main/cli/, configs/, scripts/, requirements.txt\n"
        f"  请检查上传的代码包是否完整"
    )

# 定位 CEG-WM 根目录
try:
    CEG_WM_ROOT = find_ceg_wm_root(WORK_DIR)
    print(f"✓ 找到 CEG-WM 根目录：{CEG_WM_ROOT}")
    print(f"  绝对路径：{CEG_WM_ROOT.resolve()}")
except FileNotFoundError as e:
    print(f"✗ {e}")
    # 列出实际的目录结构以帮助调试
    print("\n实际目录结构：")
    for item in WORK_DIR.iterdir():
        if item.is_dir() and not item.name.startswith('.'):
            print(f"  {item.name}/")
            for subitem in item.iterdir():
                if subitem.is_dir():
                    print(f"    {subitem.name}/")


In [ ]:

# 添加 CEG-WM 到 Python 路径
if str(CEG_WM_ROOT) not in sys.path:
    sys.path.insert(0, str(CEG_WM_ROOT))

# 验证关键模块可导入
print("验证关键模块导入...")
try:
    from main.cli import run_embed
    print("  ✓ main.cli.run_embed 导入成功")
except ImportError as e:
    print(f"  ✗ 导入失败：{e}")
    print("    这表示 CEG-WM 依赖未完整安装，将在下一步修复")


## 第 2 步：安装依赖包

安装 CEG-WM 项目本身和所有依赖。这是关键步骤！

In [ ]:

import subprocess
import sys

print("=" * 80)
print("安装依赖包")
print("=" * 80)

# 步骤 1：升级 pip
print("\n[1/3] 升级 pip...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], 
               capture_output=True)
print("  ✓ pip 已升级")

# 步骤 2：安装系统依赖
if IN_COLAB:
    print("\n[2/3] 安装系统依赖...")
    subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git", "wget", "unzip"], 
                   capture_output=True)
    print("  ✓ 系统依赖已安装")

# 步骤 3：安装 CEG-WM 项目本身（关键！）
print("\n[3/3] 安装 CEG-WM 项目...")
try:
    # 方式 A：从 pyproject.toml（推荐）
    pyproject_file = CEG_WM_ROOT / "pyproject.toml"
    requirements_file = CEG_WM_ROOT / "requirements.txt"
    
    if pyproject_file.exists():
        print(f"  从 pyproject.toml 安装：{pyproject_file}")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-e", str(CEG_WM_ROOT)],
            capture_output=True,
            text=True,
            timeout=300
        )
        if result.returncode == 0:
            print("  ✓ 项目安装成功（开发模式）")
        else:
            print(f"  ⚠ 项目安装失败，尝试备选方案")
            print(f"    错误：{result.stderr[-200:]}")
    
    elif requirements_file.exists():
        print(f"  从 requirements.txt 安装：{requirements_file}")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-r", str(requirements_file)],
            capture_output=True,
            text=True,
            timeout=300
        )
        if result.returncode == 0:
            print("  ✓ 依赖安装成功")
        else:
            print(f"  ⚠ 部分依赖安装失败，继续使用...")
    else:
        print("  ⚠ 未找到 pyproject.toml 或 requirements.txt")
        
except subprocess.TimeoutExpired:
    print("  ⚠ 安装超时（300 秒），但继续执行")
except Exception as e:
    print(f"  ⚠ 安装出错：{e}，但继续执行")

print("\n✓ 依赖安装步骤完成")


## 第 3 步：下载模型权重

真实下载 Stable Diffusion 3 模型。这是关键步骤！

In [ ]:
import time
import os
import gc
import torch
from pathlib import Path
from huggingface_hub import scan_cache_dir, snapshot_download

print("=" * 80)
print("下载模型权重")
print("=" * 80)

# 配置模型缓存目录
MODEL_CACHE_DIR = WORK_DIR / "huggingface_cache"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(MODEL_CACHE_DIR)

print(f"\n模型缓存目录：{MODEL_CACHE_DIR}")

print("\n检查 Hugging Face 认证...")
try:
    from huggingface_hub import HfApi
    api = HfApi()
    try:
        user_info = api.whoami()
        print(f"  ✓ 已认证：{user_info.get('name', 'Unknown')}")
    except Exception:
        print("  ℹ 未认证（使用匿名访问，私有/受限模型可能无法下载）")
except ImportError:
    print("  ⚠ huggingface_hub 未安装")

print("\n" + "=" * 80)
print("下载 Stable Diffusion 3.5 模型")
print("=" * 80)

MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
print(f"目标模型：{MODEL_ID}")

def check_model_cached(model_id):
    """
    功能：检查模型是否已在缓存中。
    
    Check if model is already cached locally.
    
    Args:
        model_id: Model identifier.
        
    Returns:
        Boolean indicating if model is cached.
    """
    try:
        cache_info = scan_cache_dir()
        for repo in cache_info.repos:
            if model_id in repo.repo_id:
                return True
        return False
    except Exception:
        return False

if check_model_cached(MODEL_ID):
    print(f"✓ 模型已缓存：{MODEL_ID}")
    print("  跳过下载")
else:
    print(f"\n⏳ 模型未缓存，开始下载：{MODEL_ID}")
    print("   这可能需要 5-20 分钟（取决于网络与镜像）")
    
    try:
        snapshot_path = snapshot_download(
            repo_id=MODEL_ID,
            cache_dir=str(MODEL_CACHE_DIR),
            local_files_only=False,
        )
        print("  ✓ 模型下载完成")
        print(f"  本地快照路径：{snapshot_path}")
    except Exception as e:
        error_msg = str(e)
        print(f"  ✗ 模型下载失败：{e}")
        
        if "404" in error_msg or "Entry Not Found" in error_msg:
            print("\n  ❌ 错误：无法访问模型（404）")
            print("  原因：SD3.5 可能需要先在 Hugging Face 页面授权")
        elif "401" in error_msg or "Unauthorized" in error_msg or "403" in error_msg:
            print("\n  ❌ 错误：认证失败（401/403）")
            print("  原因：HF_TOKEN 无效、缺失或未获模型访问权限")
        else:
            print("\n  解决方案：")
            print("    1. 检查网络连接")
            print("    2. 检查 HF_TOKEN 与模型访问授权")
            print("    3. 重新执行本 cell")
        
        print("\n  ⚠️ 继续执行（后续步骤可能失败）...")

print("\n清理内存...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("  ✓ GPU 缓存已清理")

print("✓ 模型准备完成")

## 第 4 步：配置选择和数据准备

选择工作流配置并准备输入数据。

In [ ]:
import json

print("=" * 80)
print("工作流配置和数据准备")
print("=" * 80)

# 选择配置文件（对齐当前仓库，仅保留 default）
CONFIG_CHOICE = "default"
print(f"\n选定配置：{CONFIG_CHOICE}")

CONFIG_FILE = CEG_WM_ROOT / "configs" / f"{CONFIG_CHOICE}.yaml"
if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"默认配置不存在：{CONFIG_FILE}")

PAPER_SPEC_FILE = CEG_WM_ROOT / "configs" / "paper_faithfulness_spec.yaml"
if PAPER_SPEC_FILE.exists():
    print(f"  ✓ 论文一致性规范存在：{PAPER_SPEC_FILE.name}")
else:
    print(f"  ⚠ 未找到论文一致性规范：{PAPER_SPEC_FILE}")

MODEL_CFG_FILE = CEG_WM_ROOT / "configs" / "model_sd3.yaml"
if MODEL_CFG_FILE.exists():
    print(f"  ✓ SD3 模型配置存在：{MODEL_CFG_FILE.name}")
else:
    print(f"  ⚠ 未找到 SD3 模型配置：{MODEL_CFG_FILE}")

print(f"  配置文件路径：{CONFIG_FILE}")
print(f"  配置文件大小：{CONFIG_FILE.stat().st_size / 1024:.1f} KB")

# 创建数据目录
data_dir = CEG_WM_ROOT / "data"
data_dir.mkdir(parents=True, exist_ok=True)

# 准备输入数据
print("\n准备输入数据...")

In [ ]:
# 第 4.5 步：paper_full_cuda 运行前预检（建议在第 5 步前执行）
import os
import shutil
import socket
from pathlib import Path

print("=" * 80)
print("运行前预检：paper_full_cuda")
print("=" * 80)

if 'CEG_WM_ROOT' not in globals():
    raise RuntimeError("请先执行前面的单元，确保 CEG_WM_ROOT 已初始化")

precheck_results = []

def _record_check(name: str, ok: bool, detail: str):
    precheck_results.append({"name": name, "ok": ok, "detail": detail})
    status = "✓" if ok else "✗"
    print(f"{status} {name}: {detail}")

# 1) CUDA / GPU 预检
try:
    import torch
    cuda_ok = bool(torch.cuda.is_available())
    if cuda_ok:
        gpu_name = torch.cuda.get_device_name(0)
        _record_check("CUDA 可用", True, f"device={gpu_name}")
    else:
        _record_check("CUDA 可用", False, "未检测到 GPU，请在 Colab 切换到 GPU Runtime")
except Exception as exc:
    _record_check("CUDA 可用", False, f"torch 检查异常: {type(exc).__name__}: {exc}")

# 2) 关键配置与脚本存在性预检
required_paths = [
    CEG_WM_ROOT / "configs" / "paper_full_cuda.yaml",
    CEG_WM_ROOT / "configs" / "paper_faithfulness_spec.yaml",
    CEG_WM_ROOT / "scripts" / "run_onefile_workflow.py",
    CEG_WM_ROOT / "scripts" / "assert_paper_mechanisms.py",
]
for path in required_paths:
    _record_check(f"文件存在: {path.name}", path.exists(), str(path))

# 3) Python 包可用性预检
required_modules = ["yaml", "huggingface_hub", "diffusers", "transformers", "safetensors"]
for module_name in required_modules:
    try:
        __import__(module_name)
        _record_check(f"模块可导入: {module_name}", True, "ok")
    except Exception as exc:
        _record_check(f"模块可导入: {module_name}", False, f"{type(exc).__name__}: {exc}")

# 4) Hugging Face 网络与访问预检（非阻塞，但强烈建议通过）
hf_ok = False
hf_detail = "未执行"
try:
    from huggingface_hub import HfApi
    socket.setdefaulttimeout(15)
    api = HfApi()
    _ = api.model_info("stabilityai/stable-diffusion-3.5-medium")
    hf_ok = True
    hf_detail = "可访问 stabilityai/stable-diffusion-3.5-medium"
except Exception as exc:
    hf_ok = False
    hf_detail = f"访问失败（可能是网络/权限问题）: {type(exc).__name__}: {exc}"
_record_check("HF 模型可访问", hf_ok, hf_detail)

# 5) 工作空间磁盘空间预检
usage = shutil.disk_usage(str(CEG_WM_ROOT))
free_gb = usage.free / (1024 ** 3)
disk_ok = free_gb >= 20.0
_record_check("磁盘剩余空间", disk_ok, f"free={free_gb:.1f} GB，建议 >= 20 GB")

# 汇总
hard_fail_names = {
    "CUDA 可用",
    "文件存在: paper_full_cuda.yaml",
    "文件存在: run_onefile_workflow.py",
    "模块可导入: diffusers",
    "模块可导入: transformers",
}
hard_fail = [item for item in precheck_results if (item["name"] in hard_fail_names and not item["ok"]) ]
soft_fail = [item for item in precheck_results if (item["name"] not in hard_fail_names and not item["ok"]) ]

print("\n" + "-" * 80)
print("预检结果汇总")
print("-" * 80)
print(f"硬性失败项数量: {len(hard_fail)}")
print(f"软性失败项数量: {len(soft_fail)}")

if hard_fail:
    print("\n✗ 预检未通过（存在硬性失败项），请先修复后再执行第 5 步。")
    for item in hard_fail:
        print(f"  - {item['name']}: {item['detail']}")
else:
    if soft_fail:
        print("\n✓ 预检通过（存在软性风险项，建议优先处理）：")
        for item in soft_fail:
            print(f"  - {item['name']}: {item['detail']}")
    else:
        print("\n✓ 预检全部通过，可以执行第 5 步。")

## 第 5-8 步：CUDA 真实模型一键完整流程（含审计与签署）

### 第 5 步
- 一键执行完整链路（embed → detect → calibrate → evaluate → audits → signoff）

### 第 6 步
- 如需单独复跑审计脚本，直接调用 `scripts/run_all_audits.py --strict`

### 第 7 步
- 如需单独复跑签署脚本，直接调用 `scripts/run_freeze_signoff.py`

### 第 8 步
- 汇总显示关键输出文件与阶段状态

**执行策略**：

- 强制要求 CUDA 可用
- 强制使用真实模型 `stabilityai/stable-diffusion-3.5-medium`
- 主流程优先调用 `scripts/run_onefile_workflow.py`，降低 Notebook 内多段逻辑出错概率

In [ ]:
import json
import sys
import subprocess
import shutil
from pathlib import Path
import torch

print("=" * 80)
print("第 5 步：CUDA + paper_full_cuda 一键完整 workflow（含 audits/signoff）")
print("=" * 80)

if 'CEG_WM_ROOT' not in globals() or 'CONFIG_CHOICE' not in globals():
    raise RuntimeError("请先执行前面的环境与配置 cell（第 1-4 步）")

if not torch.cuda.is_available():
    raise RuntimeError("当前运行时未检测到 CUDA。请在 Colab 切换到 GPU Runtime 后重试。")

gpu_name = torch.cuda.get_device_name(0)
print(f"✓ CUDA 可用，设备：{gpu_name}")

RUN_ROOT = CEG_WM_ROOT / "outputs" / "colab_run_paper_full_cuda"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print("\n准备新的运行目录...")
for name in ["records", "artifacts", "logs"]:
    target = RUN_ROOT / name
    if target.exists():
        shutil.rmtree(target)
        print(f"  已清理：{target}")

logs_dir = RUN_ROOT / "logs"
logs_dir.mkdir(parents=True, exist_ok=True)

PROJECT_PAPER_CFG = CEG_WM_ROOT / "configs" / "paper_full_cuda.yaml"
if not PROJECT_PAPER_CFG.exists():
    raise FileNotFoundError(
        f"未找到项目配置：{PROJECT_PAPER_CFG}。"
        "请确认仓库已包含 configs/paper_full_cuda.yaml。"
    )

ACTIVE_CONFIG_FILE = PROJECT_PAPER_CFG
ACTIVE_WORKFLOW_PROFILE = "paper_full_cuda"
ACTIVE_SIGNOFF_PROFILE = "paper"

print("\n一键执行参数：")
print(f"  输出目录：{RUN_ROOT}")
print(f"  运行配置（项目内）：{ACTIVE_CONFIG_FILE}")
print(f"  profile: {ACTIVE_WORKFLOW_PROFILE}")
print(f"  signoff-profile: {ACTIVE_SIGNOFF_PROFILE}")

command = [
    sys.executable,
    "scripts/run_onefile_workflow.py",
    "--cfg",
    str(ACTIVE_CONFIG_FILE),
    "--run-root",
    str(RUN_ROOT),
    "--profile",
    ACTIVE_WORKFLOW_PROFILE,
    "--signoff-profile",
    ACTIVE_SIGNOFF_PROFILE,
]

print("\n执行命令：")
print("  " + " ".join(command))

result = subprocess.run(
    command,
    cwd=str(CEG_WM_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

workflow_log = logs_dir / "workflow_execution.log"
with open(workflow_log, "w", encoding="utf-8") as f:
    f.write("COMMAND:\n")
    f.write(" ".join(command))
    f.write("\n\nRETURN_CODE:\n")
    f.write(str(result.returncode))
    f.write("\n\nSTDOUT:\n")
    f.write(result.stdout or "")
    f.write("\n\nSTDERR:\n")
    f.write(result.stderr or "")

print(f"\n返回码：{result.returncode}")
print(f"日志文件：{workflow_log}")

if result.stdout:
    print("\nSTDOUT（最后 40 行）：")
    print("\n".join(result.stdout.splitlines()[-40:]))
if result.stderr:
    print("\nSTDERR（最后 20 行）：")
    print("\n".join(result.stderr.splitlines()[-20:]))

if result.returncode != 0:
    raise RuntimeError(f"onefile workflow 执行失败，return_code={result.returncode}。请查看日志：{workflow_log}")

STAGE_STATUS = {}
STAGE_STATUS["embed"] = "ok" if (RUN_ROOT / "records" / "embed_record.json").exists() else "failed"
STAGE_STATUS["detect"] = "ok" if (RUN_ROOT / "records" / "detect_record.json").exists() else "failed"
STAGE_STATUS["calibrate"] = "ok" if (RUN_ROOT / "records" / "calibrate_record.json").exists() else "failed"
STAGE_STATUS["evaluate"] = "ok" if (RUN_ROOT / "records" / "evaluate_record.json").exists() else "failed"
STAGE_STATUS["multi_protocol"] = "ok" if (RUN_ROOT / "artifacts" / "multi_protocol_evaluation" / "artifacts" / "protocol_compare" / "compare_summary.json").exists() else "failed"
STAGE_STATUS["signoff"] = "ok" if (RUN_ROOT / "artifacts" / "signoff" / "signoff_report.json").exists() else "failed"

REQUIRED_PACKAGE_FILES = [
    RUN_ROOT / "records" / "embed_record.json",
    RUN_ROOT / "records" / "detect_record.json",
    RUN_ROOT / "records" / "calibrate_record.json",
    RUN_ROOT / "records" / "evaluate_record.json",
    RUN_ROOT / "artifacts" / "evaluation_report.json",
    RUN_ROOT / "artifacts" / "run_closure.json",
    RUN_ROOT / "artifacts" / "signoff" / "signoff_report.json",
    RUN_ROOT / "artifacts" / "multi_protocol_evaluation" / "artifacts" / "protocol_compare" / "compare_summary.json",
]

SIGNOFF_DECISION = "<absent>"
signoff_path = RUN_ROOT / "artifacts" / "signoff" / "signoff_report.json"
if signoff_path.exists():
    try:
        with open(signoff_path, "r", encoding="utf-8") as f:
            _signoff_obj = json.load(f)
        SIGNOFF_DECISION = _signoff_obj.get("freeze_signoff_decision", "<absent>")
    except Exception:
        SIGNOFF_DECISION = "<parse_failed>"

EMBED_STATUS = STAGE_STATUS.get("embed", "unknown")
DETECT_STATUS = STAGE_STATUS.get("detect", "unknown")
CALIBRATE_STATUS = STAGE_STATUS.get("calibrate", "unknown")
EVALUATE_STATUS = STAGE_STATUS.get("evaluate", "unknown")

EMBED_RUNTIME_FALLBACK_USED = False

print("\n" + "=" * 80)
print("一键 workflow 执行完成")
print("=" * 80)
print(f"阶段汇总：{STAGE_STATUS}")
print(f"Signoff 决策：{SIGNOFF_DECISION}")
print(f"结果目录：{RUN_ROOT}")

In [ ]:
print("=" * 80)
print("第 6 步：单独复跑严格审计（可选）")
print("=" * 80)

if 'CEG_WM_ROOT' not in globals() or 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步，生成 RUN_ROOT 后再复跑审计")

audit_cmd = [
    sys.executable,
    "scripts/run_all_audits.py",
    "--repo-root",
    str(CEG_WM_ROOT),
    "--strict",
]

print("执行命令：")
print("  " + " ".join(audit_cmd))

audit_result = subprocess.run(
    audit_cmd,
    cwd=str(CEG_WM_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

audit_log = RUN_ROOT / "logs" / "audits_strict_rerun.log"
with open(audit_log, "w", encoding="utf-8") as f:
    f.write("COMMAND:\n")
    f.write(" ".join(audit_cmd))
    f.write("\n\nRETURN_CODE:\n")
    f.write(str(audit_result.returncode))
    f.write("\n\nSTDOUT:\n")
    f.write(audit_result.stdout or "")
    f.write("\n\nSTDERR:\n")
    f.write(audit_result.stderr or "")

print(f"返回码：{audit_result.returncode}")
print(f"日志文件：{audit_log}")

if audit_result.stdout:
    print("\nSTDOUT（最后 30 行）：")
    print("\n".join(audit_result.stdout.splitlines()[-30:]))
if audit_result.stderr:
    print("\nSTDERR（最后 20 行）：")
    print("\n".join(audit_result.stderr.splitlines()[-20:]))

In [ ]:
print("=" * 80)
print("第 7 步：单独复跑签署脚本（可选）")
print("=" * 80)

if 'CEG_WM_ROOT' not in globals() or 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步，生成 RUN_ROOT 后再复跑签署")

signoff_cmd = [
    sys.executable,
    "scripts/run_freeze_signoff.py",
    "--run-root",
    str(RUN_ROOT),
    "--repo-root",
    str(CEG_WM_ROOT),
    "--signoff-profile",
    "paper",
]

print("执行命令：")
print("  " + " ".join(signoff_cmd))

signoff_result = subprocess.run(
    signoff_cmd,
    cwd=str(CEG_WM_ROOT),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

signoff_log = RUN_ROOT / "logs" / "signoff_rerun.log"
with open(signoff_log, "w", encoding="utf-8") as f:
    f.write("COMMAND:\n")
    f.write(" ".join(signoff_cmd))
    f.write("\n\nRETURN_CODE:\n")
    f.write(str(signoff_result.returncode))
    f.write("\n\nSTDOUT:\n")
    f.write(signoff_result.stdout or "")
    f.write("\n\nSTDERR:\n")
    f.write(signoff_result.stderr or "")

print(f"返回码：{signoff_result.returncode}")
print(f"日志文件：{signoff_log}")

if signoff_result.stdout:
    print("\nSTDOUT（最后 30 行）：")
    print("\n".join(signoff_result.stdout.splitlines()[-30:]))
if signoff_result.stderr:
    print("\nSTDERR（最后 20 行）：")
    print("\n".join(signoff_result.stderr.splitlines()[-20:]))

In [ ]:
print("=" * 80)
print("第 8 步：关键结果速查")
print("=" * 80)

if 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步")

check_items = [
    ("embed_record", RUN_ROOT / "records" / "embed_record.json"),
    ("detect_record", RUN_ROOT / "records" / "detect_record.json"),
    ("calibrate_record", RUN_ROOT / "records" / "calibrate_record.json"),
    ("evaluate_record", RUN_ROOT / "records" / "evaluate_record.json"),
    ("evaluation_report", RUN_ROOT / "artifacts" / "evaluation_report.json"),
    ("compare_summary", RUN_ROOT / "artifacts" / "multi_protocol_evaluation" / "artifacts" / "protocol_compare" / "compare_summary.json"),
    ("signoff_report", RUN_ROOT / "artifacts" / "signoff" / "signoff_report.json"),
    ("workflow_log", RUN_ROOT / "logs" / "workflow_execution.log"),
]

STAGE_STATUS = {}
for name, path in check_items:
    exists = path.exists()
    mark = "✓" if exists else "✗"
    print(f"{mark} {name}: {path}")

STAGE_STATUS["embed"] = "ok" if (RUN_ROOT / "records" / "embed_record.json").exists() else "failed"
STAGE_STATUS["detect"] = "ok" if (RUN_ROOT / "records" / "detect_record.json").exists() else "failed"
STAGE_STATUS["calibrate"] = "ok" if (RUN_ROOT / "records" / "calibrate_record.json").exists() else "failed"
STAGE_STATUS["evaluate"] = "ok" if (RUN_ROOT / "records" / "evaluate_record.json").exists() else "failed"
STAGE_STATUS["multi_protocol"] = "ok" if (RUN_ROOT / "artifacts" / "multi_protocol_evaluation" / "artifacts" / "protocol_compare" / "compare_summary.json").exists() else "failed"
STAGE_STATUS["signoff"] = "ok" if (RUN_ROOT / "artifacts" / "signoff" / "signoff_report.json").exists() else "failed"

print("\n阶段状态汇总：")
print(STAGE_STATUS)

## 第 9 步：验证工作流输出

检查工作流是否成功生成了所有必要的输出文件。

In [ ]:
import json

print("=" * 80)
print("验证工作流输出")
print("=" * 80)

if 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步，生成 RUN_ROOT 后再验证输出")

print("\n[1/4] 检查记录文件...")
records_dir = RUN_ROOT / "records"
expected_records = [
    "embed_record.json",
    "detect_record.json",
    "calibrate_record.json",
    "evaluate_record.json",
]

records_found = {}
for record_name in expected_records:
    record_path = records_dir / record_name
    exists = record_path.exists()
    records_found[record_name] = exists
    status = "✓" if exists else "✗"
    print(f"  {status} {record_name}")

print("\n[2/4] 检查产物文件...")
artifacts_dir = RUN_ROOT / "artifacts"
expected_artifacts = [
    "evaluation_report.json",
    "run_closure.json",
    "multi_protocol_evaluation/artifacts/protocol_compare/compare_summary.json",
    "signoff/signoff_report.json",
]

artifacts_found = {}
for artifact_name in expected_artifacts:
    artifact_path = artifacts_dir / artifact_name
    exists = artifact_path.exists()
    artifacts_found[artifact_name] = exists
    status = "✓" if exists else "✗"
    print(f"  {status} {artifact_name}")

print("\n[3/4] 校验 signoff 决策...")
signoff_decision = "<absent>"
signoff_report_path = artifacts_dir / "signoff" / "signoff_report.json"
if signoff_report_path.exists():
    with open(signoff_report_path, "r", encoding="utf-8") as f:
        signoff_report = json.load(f)
    signoff_decision = signoff_report.get("freeze_signoff_decision", "<absent>")
print(f"  freeze_signoff_decision: {signoff_decision}")

print("\n[4/4] 阶段状态汇总...")
if 'STAGE_STATUS' in globals():
    print(f"  阶段状态：{STAGE_STATUS}")
else:
    print("  未检测到 STAGE_STATUS（可能未执行第 5 步）")

all_required_records = all(records_found.values())
all_required_artifacts = all(artifacts_found.values())
is_signoff_allow = (signoff_decision == "ALLOW_FREEZE")

print("\n验证结论：")
print(f"  records 完整：{all_required_records}")
print(f"  artifacts 完整：{all_required_artifacts}")
print(f"  signoff 通过：{is_signoff_allow}")
print(f"  输出目录：{RUN_ROOT}")

## 第 10 步：结果打包和下载

将完整的 run_root 目录压缩并下载到本地计算机。

In [ ]:
import json
import shutil
from pathlib import Path

print("=" * 80)
print("打包和下载结果")
print("=" * 80)

if 'RUN_ROOT' not in globals():
    raise RuntimeError("请先执行第 5 步")

required_files = list(globals().get("REQUIRED_PACKAGE_FILES", []))
if not required_files:
    required_files = [
        RUN_ROOT / "records" / "embed_record.json",
        RUN_ROOT / "records" / "detect_record.json",
        RUN_ROOT / "records" / "calibrate_record.json",
        RUN_ROOT / "records" / "evaluate_record.json",
        RUN_ROOT / "artifacts" / "evaluation_report.json",
        RUN_ROOT / "artifacts" / "run_closure.json",
        RUN_ROOT / "artifacts" / "signoff" / "signoff_report.json",
        RUN_ROOT / "artifacts" / "multi_protocol_evaluation" / "artifacts" / "protocol_compare" / "compare_summary.json",
    ]

missing_files = [str(p) for p in required_files if not p.exists()]
if missing_files:
    print("\n✗ 检测到缺失必需产物，禁止打包：")
    for item in missing_files:
        print(f"  - {item}")
    raise RuntimeError("必需产物不完整，请先修复 workflow 后再打包")

signoff_report_path = RUN_ROOT / "artifacts" / "signoff" / "signoff_report.json"
signoff_decision = "<absent>"
if signoff_report_path.exists():
    with open(signoff_report_path, "r", encoding="utf-8") as f:
        signoff_obj = json.load(f)
    signoff_decision = signoff_obj.get("freeze_signoff_decision", "<absent>")

package_manifest = {
    "run_root": str(RUN_ROOT),
    "profile": globals().get("ACTIVE_WORKFLOW_PROFILE", "paper_full_cuda"),
    "signoff_profile": globals().get("ACTIVE_SIGNOFF_PROFILE", "paper"),
    "signoff_decision": signoff_decision,
    "required_files": [str(p.relative_to(RUN_ROOT)) for p in required_files],
}
manifest_path = RUN_ROOT / "artifacts" / "package_manifest.json"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(package_manifest, f, indent=2, ensure_ascii=False)
print(f"✓ 已写入打包清单：{manifest_path}")

ARCHIVE_NAME = f"ceg_wm_run_root_{CONFIG_CHOICE}_paper_full_cuda"
ARCHIVE_PATH = WORK_DIR / f"{ARCHIVE_NAME}.zip"

print(f"\n压缩目录结构...")
print(f"  源目录：{RUN_ROOT}")
print(f"  目标文件：{ARCHIVE_PATH.name}")

try:
    archive_without_ext = str(WORK_DIR / ARCHIVE_NAME)
    shutil.make_archive(
        archive_without_ext,
        "zip",
        RUN_ROOT.parent,
        RUN_ROOT.name
    )

    if ARCHIVE_PATH.exists():
        size_mb = ARCHIVE_PATH.stat().st_size / (1024 * 1024)
        print("✓ 压缩成功")
        print(f"  文件大小：{size_mb:.2f} MB")
        print(f"  完整路径：{ARCHIVE_PATH}")
    else:
        print("✗ 压缩失败（文件不存在）")
        ARCHIVE_PATH = None

except Exception as e:
    print(f"✗ 压缩过程出错：{e}")
    ARCHIVE_PATH = None

if ARCHIVE_PATH and ARCHIVE_PATH.exists():
    print(f"\n下载压缩包...")

    if IN_COLAB:
        from google.colab import files
        try:
            files.download(str(ARCHIVE_PATH))
            print(f"✓ 文件已下载：{ARCHIVE_PATH.name}")
            print("  查看浏览器的下载文件夹")
        except Exception as e:
            print(f"✗ 下载失败：{e}")
    else:
        print("本地环境提示：")
        print(f"  压缩包位置：{ARCHIVE_PATH}")
        print("  可以直接从文件系统访问或下载")
else:
    print("✗ 无法下载（压缩包创建失败）")

print("\n✓ 打包和下载步骤完成")

## 最后：执行总结报告

生成详细的执行总结。

In [ ]:
import json
from datetime import datetime

print("\n" + "=" * 80)
print("执行总结")
print("=" * 80)

if 'records_found' not in globals():
    records_found = {}
if 'artifacts_found' not in globals():
    artifacts_found = {}

runtime_config_used = str(ACTIVE_CONFIG_FILE) if 'ACTIVE_CONFIG_FILE' in globals() else str(CONFIG_FILE)
workflow_profile = globals().get("ACTIVE_WORKFLOW_PROFILE", "paper_full_cuda")
signoff_profile = globals().get("ACTIVE_SIGNOFF_PROFILE", "paper")

summary = {
    "timestamp": datetime.now().isoformat(),
    "environment": "Google Colab" if IN_COLAB else "Local Jupyter",
    "system_info": {
        "total_memory_gb": round(psutil.virtual_memory().total / (1024**3), 1),
        "available_memory_gb": round(psutil.virtual_memory().available / (1024**3), 1),
        "cpu_count": psutil.cpu_count(),
    },
    "config_used": CONFIG_CHOICE,
    "runtime_config_used": runtime_config_used,
    "workflow_profile": workflow_profile,
    "signoff_profile": signoff_profile,
    "ceg_wm_root": str(CEG_WM_ROOT),
    "run_root": str(RUN_ROOT),
    "records": {
        "embed": records_found.get("embed_record.json", False),
        "detect": records_found.get("detect_record.json", False),
        "calibrate": records_found.get("calibrate_record.json", False),
        "evaluate": records_found.get("evaluate_record.json", False),
    },
    "artifacts": {
        "evaluation_report": artifacts_found.get("evaluation_report.json", False),
        "run_closure": artifacts_found.get("run_closure.json", False),
        "compare_summary": artifacts_found.get("multi_protocol_evaluation/artifacts/protocol_compare/compare_summary.json", False),
        "signoff_report": artifacts_found.get("signoff/signoff_report.json", False),
    },
    "signoff_decision": globals().get("SIGNOFF_DECISION", "<absent>"),
    "archive": {
        "created": ARCHIVE_PATH is not None and ARCHIVE_PATH.exists(),
        "path": str(ARCHIVE_PATH) if ARCHIVE_PATH else None,
        "size_mb": round(ARCHIVE_PATH.stat().st_size / (1024 * 1024), 2) if (ARCHIVE_PATH and ARCHIVE_PATH.exists()) else None,
    },
}

print("\n元数据：")
print(f"  执行时间：{summary['timestamp']}")
print(f"  运行环境：{summary['environment']}")
print(f"  配置：{summary['config_used']}")
print(f"  运行配置：{summary['runtime_config_used']}")
print(f"  workflow profile：{summary['workflow_profile']}")
print(f"  signoff profile：{summary['signoff_profile']}")
print(f"  signoff decision：{summary['signoff_decision']}")

print("\n系统资源：")
for key, value in summary['system_info'].items():
    print(f"  {key}: {value}")

print("\n生成的记录文件：")
for stage, exists in summary['records'].items():
    status = "✓" if exists else "✗"
    print(f"  {status} {stage}_record.json")

print("\n生成的工件文件：")
for artifact, exists in summary['artifacts'].items():
    status = "✓" if exists else "✗"
    print(f"  {status} {artifact}")

print("\n压缩包信息：")
if summary['archive']['created']:
    print(f"  ✓ 已创建：{summary['archive']['path']}")
    print(f"  大小：{summary['archive']['size_mb']} MB")
else:
    print("  ✗ 压缩包创建失败")

summary_file = RUN_ROOT / "execution_summary.json"
with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"\n执行总结已保存到：{summary_file}")

print("\n" + "=" * 80)
print("✓ Colab 工作流执行完成！")
print("=" * 80)

## 配置选项参考

本 Notebook 第 5 步直接使用项目内的 `configs/paper_full_cuda.yaml`，并以 `paper_full_cuda` profile 执行。

| 配置项 | 值 | 说明 |
|---------|------|------|
| `workflow profile` | `paper_full_cuda` | 调用 `scripts/run_onefile_workflow.py` 的 profile |
| `cfg` | `configs/paper_full_cuda.yaml` | 仓库内正式 paper 配置（不再生成临时 colab yaml） |
| `model_id` | `stabilityai/stable-diffusion-3.5-medium` | 真实 SD3.5 模型 |
| `device` | `cuda` | 要求 Colab GPU Runtime |
| `signoff profile` | `paper` | 启用严格审计与签署路径 |

### 如需调整
- 如需修改模型或参数，请直接在项目配置文件中调整，或新建正式配置文件后在第 5 步改 `PROJECT_PAPER_CFG`。

## 输出结构参考

执行完成后，`run_root` 目录重点关注以下文件：

```
run_root/
├── records/
│   ├── embed_record.json
│   ├── detect_record.json
│   ├── calibrate_record.json
│   └── evaluate_record.json
├── artifacts/
│   ├── evaluation_report.json
│   ├── run_closure.json
│   ├── package_manifest.json
│   ├── multi_protocol_evaluation/
│   │   └── artifacts/protocol_compare/compare_summary.json
│   └── signoff/
│       └── signoff_report.json
└── logs/
    ├── workflow_execution.log
    ├── audits_strict_rerun.log   # 第 6 步可选复跑生成
    └── signoff_rerun.log         # 第 7 步可选复跑生成
```

### 关键输出文件说明
- **evaluation_report.json**：评估阶段输出的核心报告。
- **run_closure.json**：流程闭包与状态信息。
- **multi_protocol.../compare_summary.json**：多协议对比摘要（paper 必需）。
- **signoff/signoff_report.json**：冻结签署决策报告。
- **package_manifest.json**：打包前写入的产物清单与签署决策快照。
- **workflow_execution.log**：第 5 步一键执行日志（排错优先查看）。